In [2]:
import pandas as pd

In [3]:
DATA_PATH = "../Data/Processed/"

In [4]:
customers = pd.read_csv(
    DATA_PATH + "customers.csv",
    dtype={"customer_zip_code_prefix": str}
)

geolocation = pd.read_csv(
    DATA_PATH + "geolocation.csv"
)

orders = pd.read_csv(
    DATA_PATH + "orders.csv",
    parse_dates=[
        "order_purchase_timestamp",
        "order_approved_at",
        "order_delivered_carrier_date",
        "order_delivered_customer_date",
        "order_estimated_delivery_date"
    ]
)

order_items = pd.read_csv(
    DATA_PATH + "order_items.csv",
    parse_dates=["shipping_limit_date"]
)

order_payments = pd.read_csv(
    DATA_PATH + "order_payments.csv"
)

order_reviews = pd.read_csv(
    DATA_PATH + "order_reviews.csv",
    parse_dates=[
        "review_creation_date",
        "review_answer_timestamp"
    ]
)

products = pd.read_csv(
    DATA_PATH + "products.csv"
)

product_category_translation = pd.read_csv(
    DATA_PATH + "product_category_name_translation.csv"
)

sellers = pd.read_csv(
    DATA_PATH + "sellers.csv",
    dtype={"seller_zip_code_prefix": str}
)

In [5]:
datasets = {
    "customers": customers,
    "geolocation": geolocation,
    "orders": orders,
    "order_items": order_items,
    "order_payments": order_payments,
    "order_reviews": order_reviews,
    "products": products,
    "product_category_translation": product_category_translation,
    "sellers": sellers
}

for name, df in datasets.items():
    print(f"{name}: {df.shape}")

customers: (99441, 5)
geolocation: (738332, 5)
orders: (99441, 8)
order_items: (112650, 8)
order_payments: (103886, 5)
order_reviews: (99224, 7)
products: (32951, 9)
product_category_translation: (71, 2)
sellers: (3095, 4)


In [6]:
orders.dtypes

order_id                                    str
customer_id                                 str
order_status                                str
order_purchase_timestamp         datetime64[us]
order_approved_at                datetime64[us]
order_delivered_carrier_date     datetime64[us]
order_delivered_customer_date    datetime64[us]
order_estimated_delivery_date    datetime64[us]
dtype: object

In [7]:
order_items.dtypes

order_id                          str
order_item_id                   int64
product_id                        str
seller_id                         str
shipping_limit_date    datetime64[us]
price                         float64
freight_value                 float64
item_total                    float64
dtype: object

In [8]:
customers.dtypes

customer_id                 str
customer_unique_id          str
customer_zip_code_prefix    str
customer_city               str
customer_state              str
dtype: object

In [9]:
sellers.dtypes

seller_id                 str
seller_zip_code_prefix    str
seller_city               str
seller_state              str
dtype: object

                 ORDER-LEVEL TABLES
            ┌──────────┬─────────────┐
            │          │             │
         orders     payments      reviews
            │          │             │
            └──────────┴─────────────┘
                       │
                aggregate where needed
                       │
                       ▼
                 ORDER_ITEMS
              ⭐ MAIN GRAIN ⭐
                       │
                 ┌─────┴─────┐
                 ▼           ▼
             products      sellers
                 │
                 ▼
              category

In [10]:
order_items[["order_id", "order_item_id"]].duplicated().sum()

np.int64(0)

In [11]:
order_items.groupby("order_id").size().describe()

count    98666.000000
mean         1.141731
std          0.538452
min          1.000000
25%          1.000000
50%          1.000000
75%          1.000000
max         21.000000
dtype: float64

In [12]:
orders["order_id"].duplicated().sum()

np.int64(0)

In [13]:
order_items["order_id"].isin(orders["order_id"]).value_counts()

order_id
True    112650
Name: count, dtype: int64

## Building the Item-Level Analytical Dataset

In [14]:
item_analysis = order_items.copy()

In [15]:
item_analysis.shape

(112650, 8)

In [16]:
orders.columns

Index(['order_id', 'customer_id', 'order_status', 'order_purchase_timestamp',
       'order_approved_at', 'order_delivered_carrier_date',
       'order_delivered_customer_date', 'order_estimated_delivery_date'],
      dtype='str')

In [17]:
item_analysis = item_analysis.merge(
    orders[
        [
            "order_id",
            "customer_id",
            "order_status",
            "order_purchase_timestamp",
            "order_approved_at",
            "order_delivered_carrier_date",
            "order_delivered_customer_date",
            "order_estimated_delivery_date"
        ]
    ],
    on="order_id",
    how="left"
)

In [18]:
item_analysis.shape

(112650, 15)

In [19]:
products.columns

Index(['product_id', 'product_category_name', 'product_name_lenght',
       'product_description_lenght', 'product_photos_qty', 'product_weight_g',
       'product_length_cm', 'product_height_cm', 'product_width_cm'],
      dtype='str')

In [20]:
item_analysis = item_analysis.merge(
    products[
        [
            "product_id",
            "product_category_name",
            "product_name_lenght",
            "product_description_lenght",
            "product_photos_qty",
            "product_weight_g",
            "product_length_cm",
            "product_height_cm",
            "product_width_cm"
        ]
    ],
    on="product_id",
    how="left"
)

In [21]:
item_analysis.shape

(112650, 23)

In [22]:
sellers.columns

Index(['seller_id', 'seller_zip_code_prefix', 'seller_city', 'seller_state'], dtype='str')

In [23]:
item_analysis = item_analysis.merge(
    sellers[
        [
            "seller_id",
            "seller_zip_code_prefix",
            "seller_city",
            "seller_state"
        ]
    ],
    on="seller_id",
    how="left"
)

In [24]:
item_analysis.shape

(112650, 26)

In [25]:
order_payments.groupby("order_id").size().describe()

count    99440.000000
mean         1.044710
std          0.381166
min          1.000000
25%          1.000000
50%          1.000000
75%          1.000000
max         29.000000
dtype: float64

In [26]:
order_payments.groupby("order_id")["payment_type"].nunique().value_counts().sort_index()

payment_type
1    97194
2     2246
Name: count, dtype: int64

In [27]:
order_payments[
    order_payments.groupby("order_id")["payment_type"].transform("nunique") > 1
].head(10)

,order_id,payment_sequential,payment_type,payment_installments,payment_value
25,5cfd514482e22bc992e7693f0e3e8df7,2,voucher,1,45.17
35,b2bb080b6bc860118a246fd9b6fad6da,1,credit_card,1,173.84
75,3689194c14ad4e2e7361ebd1df0e77b0,2,voucher,1,57.53
84,723e462ce1ee50e024887c0b403130f3,1,credit_card,1,13.80
102,21b8b46679ea6482cbf911d960490048,2,voucher,1,43.12
121,ea9184ad433a404df1d72fa0a8764232,4,voucher,1,22.49
161,c61e3efc183db6b89f5556efb5aa8797,1,credit_card,1,45.50
164,487c1451b8fd7347d0e80e5aca887e91,2,voucher,1,30.00
166,65c863d90026e9dd4888e28d2135c983,2,voucher,1,50.00
182,79b63d1f9cf0257b33b04fbfb0847cba,2,voucher,1,76.90


In [28]:
order_payments[
    order_payments["order_id"] == "5cfd514482e22bc992e7693f0e3e8df7"
]

,order_id,payment_sequential,payment_type,payment_installments,payment_value
25,5cfd514482e22bc992e7693f0e3e8df7,2,voucher,1,45.17
57742,5cfd514482e22bc992e7693f0e3e8df7,1,credit_card,4,665.41


In [29]:
(
    order_payments.groupby("order_id")["payment_sequential"].max()
    != order_payments.groupby("order_id").size()
).sum()

np.int64(80)

In [30]:
payment_check = (
    order_payments.groupby("order_id")
    .agg(
        max_sequence=("payment_sequential", "max"),
        payment_rows=("payment_sequential", "size")
    )
    .reset_index()
)

payment_check[
    payment_check["max_sequence"] != payment_check["payment_rows"]
].head(10)

,order_id,max_sequence,payment_rows
242,00ac05fe0fc047c54418098eb64e3aaa,2,1
2133,056c68d093c100017aab1f00f260705c,2,1
2518,0668d086b3ae41adde3aed3dacbc8fae,2,1
4111,0a7d898c6305101e69e9f5a05cd0130d,2,1
5123,0d0e88a418636d40ea780339701db49c,2,1
6196,0fdb6f73fab7005c84d4a43d8a93682b,2,1
8396,159da9914b51de617fe80162939965c5,2,1
8675,166b01d0fce75a595b92ad9e84e0843b,2,1
9538,1896e8b8fd196151559b02e31e98ce89,2,1
10244,1a57108394169c0b47d8f876acc9ba2d,2,1


In [31]:
payment_summary = (
    order_payments
    .groupby("order_id")
    .agg(
        total_payment_value=("payment_value", "sum"),
        payment_types=("payment_type", lambda x: ", ".join(sorted(x.unique()))),
        max_payment_installments=("payment_installments", "max"),
        payment_count=("payment_value", "count")
    )
    .reset_index()
)

In [32]:
payment_summary.shape

(99440, 5)

In [33]:
payment_summary.head()

,order_id,total_payment_value,payment_types,max_payment_installments,payment_count
0,00010242fe8c5a6d1ba2dd792cb16214,72.19,credit_card,2,1
1,00018f77f2f0320c557190d7a144bdd3,259.83,credit_card,3,1
2,000229ec398224ef6ca0657da4fc703e,216.87,credit_card,5,1
3,00024acbcdf0a6daa1e931b038114c75,25.78,credit_card,2,1
4,00042b26cf59d7ce69dfabb4e55b4fd9,218.04,credit_card,3,1


In [34]:
payment_summary[
    payment_summary["order_id"] == "5cfd514482e22bc992e7693f0e3e8df7"
]

,order_id,total_payment_value,payment_types,max_payment_installments,payment_count
36023,5cfd514482e22bc992e7693f0e3e8df7,710.58,"credit_card, voucher",4,2


In [35]:
item_analysis = item_analysis.merge(
    payment_summary,
    on="order_id",
    how="left"
)

In [36]:
item_analysis.shape

(112650, 30)

In [37]:
order_reviews.groupby("order_id").size().describe()

count    98673.000000
mean         1.005584
std          0.075060
min          1.000000
25%          1.000000
50%          1.000000
75%          1.000000
max          3.000000
dtype: float64

In [38]:
order_reviews[
    order_reviews.groupby("order_id").transform("size") > 1
].sort_values("order_id").head(10)

,review_id,order_id,review_score,review_comment_title,review_comment_message,review_creation_date,review_answer_timestamp
25612,89a02c45c340aeeb1354a24e7d4b2c1e,0035246a40f520710769010f752e7507,5,NaN,NaN,2017-08-29,2017-08-30 01:59:12
22423,2a74b0559eb58fc1ff842ecc999594cb,0035246a40f520710769010f752e7507,5,NaN,Estou acostumada a comprar produtos pelo barat...,2017-08-25,2017-08-29 21:45:57
22779,ab30810c29da5da8045216f0f62652a2,013056cfe49763c6f66bda03396c5ee3,5,NaN,NaN,2018-02-22,2018-02-23 12:12:30
68633,73413b847f63e02bc752b364f6d05ee9,013056cfe49763c6f66bda03396c5ee3,4,NaN,NaN,2018-03-04,2018-03-05 17:02:00
854,830636803620cdf8b6ffaf1b2f6e92b2,0176a6846bcb3b0d3aa3116a9a768597,5,NaN,NaN,2017-12-30,2018-01-02 10:54:06
83224,d8e8c42271c8fb67b9dad95d98c8ff80,0176a6846bcb3b0d3aa3116a9a768597,5,NaN,NaN,2017-12-30,2018-01-02 10:54:47
17582,017f0e1ea6386de662cbeba299c59ad1,02355020fd0a40a0d56df9f6ff060413,1,NaN,ja reclamei varias vezes e ate hoje não sei on...,2018-03-29,2018-03-30 03:16:19
89888,0c8e7347f1cdd2aede37371543e3d163,02355020fd0a40a0d56df9f6ff060413,3,NaN,UM DOS PRODUTOS (ENTREGA02) COMPRADOS NESTE PE...,2018-03-21,2018-03-22 01:32:08
55137,61fe4e7d1ae801bbe169eb67b86c6eda,029863af4b968de1e5d6a82782e662f5,4,NaN,NaN,2017-07-19,2017-07-20 12:06:11
37911,04d945e95c788a3aa1ffbee42105637b,029863af4b968de1e5d6a82782e662f5,5,NaN,NaN,2017-07-14,2017-07-17 13:58:06


In [39]:
order_reviews["review_id"].duplicated().sum()

np.int64(814)

In [40]:
#814 → duplicate occurrences beyond the first .duplicated().sum() → 3 (A's 2nd + 3rd, B's 2nd)
#1603 → all rows involved in duplicated review_ids .duplicated(keep=False) → 5 rows (A A A B B)

order_reviews[order_reviews["review_id"].duplicated(keep=False)].shape

(1603, 7)

In [41]:
order_reviews[
    order_reviews["review_id"].duplicated(keep=False)
].sort_values("review_id").head(20)

,review_id,order_id,review_score,review_comment_title,review_comment_message,review_creation_date,review_answer_timestamp
46678,00130cbe1f9d422698c812ed8ded1919,dfcdfc43867d1c1381bfaf62d6b9c195,1,NaN,"O cartucho ""original HP"" 60XL não é reconhecid...",2018-03-07,2018-03-20 18:08:23
29841,00130cbe1f9d422698c812ed8ded1919,04a28263e085d399c97ae49e0b477efa,1,NaN,"O cartucho ""original HP"" 60XL não é reconhecid...",2018-03-07,2018-03-20 18:08:23
90677,0115633a9c298b6a98bcbe4eee75345f,78a4201f58af3463bdab842eea4bc801,5,NaN,NaN,2017-09-21,2017-09-26 03:27:47
63193,0115633a9c298b6a98bcbe4eee75345f,0c9850b2c179c1ef60d2855e2751d1fa,5,NaN,NaN,2017-09-21,2017-09-26 03:27:47
92876,0174caf0ee5964646040cd94e15ac95e,f93a732712407c02dce5dd5088d0f47b,1,NaN,Produto entregue dentro de embalagem do fornec...,2018-03-07,2018-03-08 03:00:53
57280,0174caf0ee5964646040cd94e15ac95e,74db91e33b4e1fd865356c89a61abf1f,1,NaN,Produto entregue dentro de embalagem do fornec...,2018-03-07,2018-03-08 03:00:53
54832,017808d29fd1f942d97e50184dfb4c13,8daaa9e99d60fbba579cc1c3e3bfae01,5,NaN,NaN,2018-03-02,2018-03-05 01:43:30
99167,017808d29fd1f942d97e50184dfb4c13,b1461c8882153b5fe68307c46a506e39,5,NaN,NaN,2018-03-02,2018-03-05 01:43:30
20621,0254bd905dc677a6078990aad3331a36,5bf226cf882c5bf4247f89a97c86f273,1,NaN,O pedido consta de 2 produtos e até agora rece...,2017-09-09,2017-09-13 09:52:44
96080,0254bd905dc677a6078990aad3331a36,331b367bdd766f3d1cf518777317b5d9,1,NaN,O pedido consta de 2 produtos e até agora rece...,2017-09-09,2017-09-13 09:52:44


In [42]:
order_reviews.groupby("order_id")["review_score"].nunique().value_counts().sort_index()

review_score
1    98471
2      202
Name: count, dtype: int64

In [43]:
order_reviews["review_score"].isna().sum()

np.int64(0)

In [44]:
review_summary = (
    order_reviews
    .groupby("order_id")
    .agg(
        review_count=("review_id", "count"),
        review_score_avg=("review_score", "mean"),
        review_score_min=("review_score", "min"),
        review_score_max=("review_score", "max")
    )
    .reset_index()
)

In [45]:
review_summary.shape

(98673, 5)

In [46]:
review_summary.head()

,order_id,review_count,review_score_avg,review_score_min,review_score_max
0,00010242fe8c5a6d1ba2dd792cb16214,1,5.0,5,5
1,00018f77f2f0320c557190d7a144bdd3,1,4.0,4,4
2,000229ec398224ef6ca0657da4fc703e,1,5.0,5,5
3,00024acbcdf0a6daa1e931b038114c75,1,4.0,4,4
4,00042b26cf59d7ce69dfabb4e55b4fd9,1,5.0,5,5


In [47]:
review_summary[
    review_summary["review_count"] > 1
].head(10)

,order_id,review_count,review_score_avg,review_score_min,review_score_max
84,0035246a40f520710769010f752e7507,2,5.000000,5,5
461,013056cfe49763c6f66bda03396c5ee3,2,4.500000,4,5
556,0176a6846bcb3b0d3aa3116a9a768597,2,5.000000,5,5
835,02355020fd0a40a0d56df9f6ff060413,2,2.000000,1,3
985,029863af4b968de1e5d6a82782e662f5,2,4.500000,4,5
1092,02e0b68852217f5715fb9cc885829454,2,4.000000,4,4
1103,02e723e8edb4a123d414f56cc9c4665e,2,5.000000,5,5
1272,03515a836bb855b03f7df9dee520a8fc,2,5.000000,5,5
1455,03c939fd7fd3b38f8485a0f95798f1f6,3,3.333333,3,4
1510,03eba6d9fef8f5b3e811d4b5a7cca9cd,2,4.500000,4,5


In [48]:
item_analysis = item_analysis.merge(
    review_summary,
    on="order_id",
    how="left"
)

In [49]:
item_analysis.shape

(112650, 34)

In [50]:
customers["customer_id"].duplicated().sum()

np.int64(0)

In [51]:
item_analysis = item_analysis.merge(
    customers[
        [
            "customer_id",
            "customer_unique_id",
            "customer_zip_code_prefix",
            "customer_city",
            "customer_state"
        ]
    ],
    on="customer_id",
    how="left"
)

In [52]:
item_analysis.shape

(112650, 38)

In [53]:
product_category_translation["product_category_name"].duplicated().sum()

np.int64(0)

In [54]:
item_analysis = item_analysis.merge(
    product_category_translation[
        [
            "product_category_name",
            "product_category_name_english"
        ]
    ],
    on="product_category_name",
    how="left"
)

In [55]:
item_analysis.shape

(112650, 39)

In [56]:
item_analysis[["order_id", "order_item_id"]].duplicated().sum()

np.int64(0)

In [57]:
geolocation.groupby("geolocation_zip_code_prefix").size().describe()

count    19015.000000
mean        38.828925
std         50.875735
min          1.000000
25%          8.000000
50%         23.000000
75%         49.000000
max        779.000000
dtype: float64

In [58]:
geolocation_summary = (
    geolocation
    .groupby("geolocation_zip_code_prefix")
    .agg(
        latitude=("geolocation_lat", "mean"),
        longitude=("geolocation_lng", "mean")
    )
    .reset_index()
)

In [59]:
geolocation_summary.shape

(19015, 3)

In [60]:
item_analysis.shape

(112650, 39)

In [61]:
item_analysis[["order_id", "order_item_id"]].duplicated().sum()

np.int64(0)

In [62]:
item_analysis.columns

Index(['order_id', 'order_item_id', 'product_id', 'seller_id',
       'shipping_limit_date', 'price', 'freight_value', 'item_total',
       'customer_id', 'order_status', 'order_purchase_timestamp',
       'order_approved_at', 'order_delivered_carrier_date',
       'order_delivered_customer_date', 'order_estimated_delivery_date',
       'product_category_name', 'product_name_lenght',
       'product_description_lenght', 'product_photos_qty', 'product_weight_g',
       'product_length_cm', 'product_height_cm', 'product_width_cm',
       'seller_zip_code_prefix', 'seller_city', 'seller_state',
       'total_payment_value', 'payment_types', 'max_payment_installments',
       'payment_count', 'review_count', 'review_score_avg', 'review_score_min',
       'review_score_max', 'customer_unique_id', 'customer_zip_code_prefix',
       'customer_city', 'customer_state', 'product_category_name_english'],
      dtype='str')

In [63]:
item_analysis.isna().sum().sort_values(ascending=False)

order_delivered_customer_date    2454
product_category_name_english    1627
product_name_lenght              1603
product_description_lenght       1603
product_photos_qty               1603
product_category_name            1603
order_delivered_carrier_date     1194
review_score_max                  942
review_score_avg                  942
review_score_min                  942
review_count                      942
product_height_cm                  18
product_width_cm                   18
product_weight_g                   18
product_length_cm                  18
order_approved_at                  15
payment_count                       3
payment_types                       3
total_payment_value                 3
max_payment_installments            3
product_id                          0
order_id                            0
order_item_id                       0
freight_value                       0
price                               0
shipping_limit_date                 0
seller_id   

In [64]:
print("Orders missing:", 
      (~item_analysis["order_id"].isin(orders["order_id"])).sum())

print("Products missing:",
      (~item_analysis["product_id"].dropna().isin(products["product_id"])).sum())

print("Sellers missing:",
      (~item_analysis["seller_id"].isin(sellers["seller_id"])).sum())

print("Customers missing:",
      (~item_analysis["customer_id"].isin(customers["customer_id"])).sum())

Orders missing: 0
Products missing: 0
Sellers missing: 0
Customers missing: 0


In [65]:
item_analysis.dtypes

order_id                                    str
order_item_id                             int64
product_id                                  str
seller_id                                   str
shipping_limit_date              datetime64[us]
price                                   float64
freight_value                           float64
item_total                              float64
customer_id                                 str
order_status                                str
order_purchase_timestamp         datetime64[us]
order_approved_at                datetime64[us]
order_delivered_carrier_date     datetime64[us]
order_delivered_customer_date    datetime64[us]
order_estimated_delivery_date    datetime64[us]
product_category_name                       str
product_name_lenght                     float64
product_description_lenght              float64
product_photos_qty                      float64
product_weight_g                        float64
product_length_cm                       

In [66]:
print("Negative prices:", (item_analysis["price"] < 0).sum())
print("Negative freight:", (item_analysis["freight_value"] < 0).sum())
print("Negative item totals:", (item_analysis["item_total"] < 0).sum())
print("Negative payments:", (item_analysis["total_payment_value"] < 0).sum())

Negative prices: 0
Negative freight: 0
Negative item totals: 0
Negative payments: 0


In [67]:
item_analysis["order_item_count"] = (
    item_analysis.groupby("order_id")["order_item_id"]
    .transform("count")
)

In [68]:
item_analysis["order_item_count"].describe()

count    112650.000000
mean          1.395668
std           1.120101
min           1.000000
25%           1.000000
50%           1.000000
75%           1.000000
max          21.000000
Name: order_item_count, dtype: float64

In [69]:
item_analysis["order_total_value"] = (
    item_analysis.groupby("order_id")["item_total"]
    .transform("sum")
)

In [70]:
item_analysis["order_total_value"].describe()

count    112650.000000
mean        180.244628
std         272.829911
min           9.590000
25%          65.620000
50%         114.440000
75%         195.337500
max       13664.080000
Name: order_total_value, dtype: float64

In [71]:
item_analysis["order_total_freight"] = (
    item_analysis.groupby("order_id")["freight_value"]
    .transform("sum")
)

In [72]:
item_analysis["order_total_freight"].describe()

count    112650.000000
mean         27.285033
std          33.218827
min           0.000000
25%          14.292500
50%          18.160000
75%          29.220000
max        1794.960000
Name: order_total_freight, dtype: float64

In [73]:
item_analysis["delivery_days"] = (
    item_analysis["order_delivered_customer_date"]
    - item_analysis["order_purchase_timestamp"]
).dt.total_seconds() / (24 * 60 * 60)

In [74]:
item_analysis["delivery_days"].describe()

count    110196.000000
mean         12.472631
std           9.445655
min           0.533414
25%           6.736262
50%          10.184282
75%          15.541097
max         209.628611
Name: delivery_days, dtype: float64

In [75]:
item_analysis["estimated_delivery_days"] = (
    item_analysis["order_estimated_delivery_date"]
    - item_analysis["order_purchase_timestamp"]
).dt.total_seconds() / (24 * 60 * 60)

In [76]:
item_analysis["estimated_delivery_days"].describe()

count    112650.000000
mean         23.835035
std           8.887306
min           2.008009
25%          18.370035
50%          23.255880
75%          28.471458
max         155.135463
Name: estimated_delivery_days, dtype: float64

In [77]:
item_analysis["delivery_delay_days"] = (
    item_analysis["delivery_days"]
    - item_analysis["estimated_delivery_days"]
)

In [78]:
item_analysis["delivery_delay_days"].describe()

count    110196.000000
mean        -11.333149
std          10.162342
min        -146.016123
25%         -16.321042
50%         -12.046788
75%          -6.476803
max         188.975081
Name: delivery_delay_days, dtype: float64

In [81]:
import numpy as np

In [82]:
item_analysis["is_delivered_late"] = np.where(
    item_analysis["delivery_delay_days"].isna(),
    np.nan,
    (item_analysis["delivery_delay_days"] > 0).astype(int)
)

In [83]:
item_analysis["is_delivered_late"].value_counts(dropna=False)

is_delivered_late
0.0    101481
1.0      8715
NaN      2454
Name: count, dtype: int64

In [84]:
conditions = [
    item_analysis["delivery_delay_days"].isna(),
    item_analysis["delivery_delay_days"] < 0,
    item_analysis["delivery_delay_days"] == 0,
    item_analysis["delivery_delay_days"] > 0
]

choices = [
    "not_delivered",
    "delivered_early",
    "delivered_on_time",
    "delivered_late"
]

item_analysis["delivery_status"] = np.select(
    conditions,
    choices,
    default="unknown"
)

In [85]:
item_analysis["delivery_status"].value_counts()

delivery_status
delivered_early    101481
delivered_late       8715
not_delivered        2454
Name: count, dtype: int64

In [86]:
item_analysis["customer_order_count"] = (
    item_analysis.groupby("customer_id")["order_id"]
    .transform("nunique")
)

In [87]:
item_analysis["customer_order_count"].describe()

count    112650.0
mean          1.0
std           0.0
min           1.0
25%           1.0
50%           1.0
75%           1.0
max           1.0
Name: customer_order_count, dtype: float64

In [93]:
item_analysis.drop(columns="customer_order_count", inplace=True)

In [94]:
item_analysis["customer_order_count"] = (
    item_analysis.groupby("customer_unique_id")["order_id"]
    .transform("nunique")
)

In [95]:
item_analysis["customer_order_count"].describe()

count    112650.000000
mean          1.082832
std           0.393762
min           1.000000
25%           1.000000
50%           1.000000
75%           1.000000
max          16.000000
Name: customer_order_count, dtype: float64

In [96]:
item_analysis.groupby("customer_unique_id")["order_id"].nunique().value_counts().sort_index()

order_id
1     92507
2      2673
3       192
4        29
5         9
6         5
7         3
9         1
16        1
Name: count, dtype: int64

In [97]:
item_analysis["customer_total_spend"] = (
    item_analysis.groupby("customer_unique_id")["item_total"]
    .transform("sum")
)

In [98]:
item_analysis["customer_total_spend"].describe()

count    112650.000000
mean        193.516281
std         294.148118
min           9.590000
25%          68.050000
50%         120.065000
75%         209.080000
max       13664.080000
Name: customer_total_spend, dtype: float64

In [99]:
item_analysis["customer_is_repeat"] = (
    item_analysis["customer_order_count"] > 1
).astype(int)

In [100]:
item_analysis["customer_is_repeat"].value_counts()

customer_is_repeat
0    105167
1      7483
Name: count, dtype: int64

In [101]:
item_analysis.groupby("customer_unique_id")["customer_is_repeat"].first().value_counts()

customer_is_repeat
0    92507
1     2913
Name: count, dtype: int64

In [102]:
item_analysis["seller_item_count"] = (
    item_analysis.groupby("seller_id")["order_item_id"]
    .transform("count")
)

In [103]:
item_analysis["seller_item_count"].describe()

count    112650.000000
mean        426.603444
std         563.122004
min           1.000000
25%          56.000000
50%         173.000000
75%         529.000000
max        2033.000000
Name: seller_item_count, dtype: float64

In [104]:
item_analysis["seller_order_count"] = (
    item_analysis.groupby("seller_id")["order_id"]
    .transform("nunique")
)

In [105]:
item_analysis["seller_order_count"].describe()

count    112650.000000
mean        374.260852
std         490.294546
min           1.000000
25%          50.000000
50%         151.000000
75%         473.000000
max        1854.000000
Name: seller_order_count, dtype: float64

In [106]:
item_analysis["seller_total_sales"] = (
    item_analysis.groupby("seller_id")["item_total"]
    .transform("sum")
)

In [107]:
item_analysis["seller_total_sales"].describe()

count    112650.000000
mean      48570.067114
std       64747.900295
min          12.220000
25%        6255.770000
50%       17220.940000
75%       54722.290000
max      249640.700000
Name: seller_total_sales, dtype: float64

In [108]:
item_analysis[[
    "seller_id",
    "seller_item_count",
    "seller_order_count",
    "seller_total_sales"
]].drop_duplicates("seller_id").describe()

,seller_item_count,seller_order_count,seller_total_sales
count,3095.000000,3095.000000,3095.000000
mean,36.397415,32.313409,5119.080207
std,119.193461,105.139763,15922.192727
min,1.000000,1.000000,12.220000
25%,2.000000,2.000000,269.735000
50%,8.000000,6.000000,996.840000
75%,24.000000,21.500000,3964.530000
max,2033.000000,1854.000000,249640.700000


In [110]:
item_analysis.shape

(112650, 53)

In [111]:
item_analysis.columns.tolist()

['order_id',
 'order_item_id',
 'product_id',
 'seller_id',
 'shipping_limit_date',
 'price',
 'freight_value',
 'item_total',
 'customer_id',
 'order_status',
 'order_purchase_timestamp',
 'order_approved_at',
 'order_delivered_carrier_date',
 'order_delivered_customer_date',
 'order_estimated_delivery_date',
 'product_category_name',
 'product_name_lenght',
 'product_description_lenght',
 'product_photos_qty',
 'product_weight_g',
 'product_length_cm',
 'product_height_cm',
 'product_width_cm',
 'seller_zip_code_prefix',
 'seller_city',
 'seller_state',
 'total_payment_value',
 'payment_types',
 'max_payment_installments',
 'payment_count',
 'review_count',
 'review_score_avg',
 'review_score_min',
 'review_score_max',
 'customer_unique_id',
 'customer_zip_code_prefix',
 'customer_city',
 'customer_state',
 'product_category_name_english',
 'order_item_count',
 'order_total_value',
 'order_total_freight',
 'delivery_days',
 'estimated_delivery_days',
 'delivery_delay_days',
 'is_deli

In [112]:
item_analysis["freight_percentage"] = (
    item_analysis["freight_value"] / item_analysis["price"] * 100
)

In [113]:
item_analysis["customer_avg_order_value"] = (
    item_analysis["customer_total_spend"]
    / item_analysis["customer_order_count"]
)

In [114]:
item_analysis["seller_avg_order_value"] = (
    item_analysis["seller_total_sales"]
    / item_analysis["seller_order_count"]
)

In [115]:
item_analysis["order_year"] = (
    item_analysis["order_purchase_timestamp"].dt.year
)

In [116]:
item_analysis["order_month"] = (
    item_analysis["order_purchase_timestamp"].dt.month
)

In [117]:
item_analysis[[
    "freight_percentage",
    "customer_avg_order_value",
    "seller_avg_order_value",
    "order_year",
    "order_month"
]].describe()

,freight_percentage,customer_avg_order_value,seller_avg_order_value,order_year,order_month
count,112650.000000,112650.000000,112650.000000,112650.000000,112650.000000
mean,32.086355,179.623181,158.783528,2017.541909,6.026835
std,34.989368,270.835108,166.476266,0.504792,3.232541
min,0.000000,9.590000,12.220000,2016.000000,1.000000
25%,13.403366,66.372500,81.588873,2017.000000,3.000000
50%,23.135624,114.790000,124.788407,2018.000000,6.000000
75%,39.303621,194.695000,180.701667,2018.000000,8.000000
max,2623.529412,13664.080000,6922.210000,2018.000000,12.000000


In [118]:
item_analysis.shape

(112650, 58)

In [119]:
item_analysis[
    ["order_id", "product_id", "price", "freight_value", "freight_percentage"]
].sort_values(
    "freight_percentage", ascending=False
).head(10)

,order_id,product_id,price,freight_value,freight_percentage
87081,c5bdd8ef3c0ec420232e668302179113,8a3254bee785a526d548a81a9bc3c9be,0.85,22.30,2623.529412
48625,6e864b3f0ec71031117ad4cf46b7f2a1,8a3254bee785a526d548a81a9bc3c9be,0.85,18.23,2144.705882
27652,3ee6513ae7ea23bdfab5b9ab60bffcb5,8a3254bee785a526d548a81a9bc3c9be,0.85,18.23,2144.705882
110535,fb265b2dc558a56445dfc48f8224e201,baf25ed4f8f70238cc87230379471454,9.90,121.22,1224.444444
94495,d642656598ae928a250620315d19e87e,b07fffe072c9adc235a35d8da7c0584d,4.99,37.04,742.284569
57312,8272b63d03f5f79c56e9e4120aec44ef,270516a3f41dc035aa87d220228f844c,1.20,7.89,657.500000
57303,8272b63d03f5f79c56e9e4120aec44ef,05b515fdc76e888aada3c6d66c201dff,1.20,7.89,657.500000
57299,8272b63d03f5f79c56e9e4120aec44ef,05b515fdc76e888aada3c6d66c201dff,1.20,7.89,657.500000
57297,8272b63d03f5f79c56e9e4120aec44ef,270516a3f41dc035aa87d220228f844c,1.20,7.89,657.500000
57300,8272b63d03f5f79c56e9e4120aec44ef,05b515fdc76e888aada3c6d66c201dff,1.20,7.89,657.500000


In [120]:
feature_audit = pd.DataFrame({
    "column": item_analysis.columns,
    "dtype": item_analysis.dtypes.astype(str).values,
    "missing": item_analysis.isna().sum().values,
    "unique_values": item_analysis.nunique().values
})

feature_audit

,column,dtype,missing,unique_values
0,order_id,str,0,98666
1,order_item_id,int64,0,21
2,product_id,str,0,32951
3,seller_id,str,0,3095
4,shipping_limit_date,datetime64[us],0,93318
5,price,float64,0,5968
6,freight_value,float64,0,6999
7,item_total,float64,0,27029
8,customer_id,str,0,98666
9,order_status,str,0,7


In [121]:
feature_audit.shape

(58, 4)

In [ ]:
feature_audit["dtype"].value_counts()

dtype
float64           28
str               16
int64              6
datetime64[us]     6
int32              2
Name: count, dtype: int64

In [125]:
item_analysis.duplicated().sum()

np.int64(0)

In [126]:
item_analysis.duplicated().sum()

np.int64(0)

In [127]:
item_analysis.duplicated(
    subset=["order_id", "order_item_id"]
).sum()

np.int64(0)

In [128]:
item_analysis[[
    "order_id",
    "order_item_id",
    "product_id",
    "seller_id",
    "customer_id",
    "customer_unique_id"
]].nunique()

order_id              98666
order_item_id            21
product_id            32951
seller_id              3095
customer_id           98666
customer_unique_id    95420
dtype: int64

In [129]:
feature_audit[
    feature_audit["unique_values"] <= 1
]

,column,dtype,missing,unique_values


In [130]:
item_analysis[[
    "item_total",
    "order_item_count",
    "order_total_value",
    "order_total_freight",
    "delivery_days",
    "estimated_delivery_days",
    "delivery_delay_days",
    "customer_order_count",
    "customer_total_spend",
    "customer_avg_order_value",
    "seller_item_count",
    "seller_order_count",
    "seller_total_sales",
    "seller_avg_order_value",
    "freight_percentage"
]].describe().T

,count,mean,std,min,25%,50%,75%,max
item_total,112650.0,140.644059,190.724394,6.080000,55.220000,92.320000,157.937500,6929.310000
order_item_count,112650.0,1.395668,1.120101,1.000000,1.000000,1.000000,1.000000,21.000000
order_total_value,112650.0,180.244628,272.829911,9.590000,65.620000,114.440000,195.337500,13664.080000
order_total_freight,112650.0,27.285033,33.218827,0.000000,14.292500,18.160000,29.220000,1794.960000
delivery_days,110196.0,12.472631,9.445655,0.533414,6.736262,10.184282,15.541097,209.628611
estimated_delivery_days,112650.0,23.835035,8.887306,2.008009,18.370035,23.255880,28.471458,155.135463
delivery_delay_days,110196.0,-11.333149,10.162342,-146.016123,-16.321042,-12.046788,-6.476803,188.975081
customer_order_count,112650.0,1.082832,0.393762,1.000000,1.000000,1.000000,1.000000,16.000000
customer_total_spend,112650.0,193.516281,294.148118,9.590000,68.050000,120.065000,209.080000,13664.080000
customer_avg_order_value,112650.0,179.623181,270.835108,9.590000,66.372500,114.790000,194.695000,13664.080000


In [131]:
item_analysis[
    [
        "order_id",
        "order_purchase_timestamp",
        "order_delivered_customer_date",
        "order_estimated_delivery_date",
        "delivery_days",
        "estimated_delivery_days",
        "delivery_delay_days"
    ]
].sort_values("delivery_delay_days").head(10)

,order_id,order_purchase_timestamp,order_delivered_customer_date,order_estimated_delivery_date,delivery_days,estimated_delivery_days,delivery_delay_days
2682,0607f0efea4b566f1eb8f7d3c2397320,2018-03-06 09:47:07,2018-03-09 23:36:47,2018-08-03,3.576157,149.592280,-146.016123
2683,0607f0efea4b566f1eb8f7d3c2397320,2018-03-06 09:47:07,2018-03-09 23:36:47,2018-08-03,3.576157,149.592280,-146.016123
87724,c72727d29cde4cf870d569bf65edabfd,2017-02-07 18:01:15,2017-02-14 14:27:45,2017-07-04,6.851736,146.249132,-139.397396
105104,eec7f369423b033e549c02f3c5381205,2018-02-06 20:44:56,2018-02-27 16:35:43,2018-07-12,20.826933,155.135463,-134.308530
85729,c2bb89b5c1dd978d507284be78a04cb2,2017-05-23 22:28:36,2017-06-09 13:35:54,2017-10-11,16.630069,140.063472,-123.433403
85730,c2bb89b5c1dd978d507284be78a04cb2,2017-05-23 22:28:36,2017-06-09 13:35:54,2017-10-11,16.630069,140.063472,-123.433403
28492,40dc2ba6f322a17626aac6244332828c,2017-10-05 21:39:05,2017-10-13 13:49:07,2018-01-30,7.673634,116.097859,-108.424225
11622,1a695d543b7302aa9446c8d5fbd632bf,2017-12-16 10:32:49,2017-12-28 22:18:23,2018-03-22,12.489977,95.560544,-83.070567
25446,39e0115911bf404857e14baa7f097feb,2018-01-20 07:48:16,2018-02-01 01:52:34,2018-04-25,11.752986,94.674815,-82.921829
37726,559eea5a72341a4c82dbce9884277cb7,2017-11-17 10:24:03,2017-11-30 16:12:50,2018-02-16,13.242211,90.566632,-77.324421


In [133]:
item_analysis[
    [
        "order_id",
        "order_purchase_timestamp",
        "order_estimated_delivery_date"
    ]
].sort_values("order_estimated_delivery_date", ascending=False).head(20)

,order_id,order_purchase_timestamp,order_estimated_delivery_date
5142,0ba4f7e89d304a02edca85087abfd14c,2018-08-13 22:51:49,2018-10-25
49637,70c5ccc21360b2f9c8f0a0decfb4e41d,2018-08-14 19:46:25,2018-10-23
86974,c583b6157fc73464387efe2ff82ee1a4,2018-08-20 21:07:24,2018-10-17
19432,2c9ef1099d75918095356bda34b2d4c0,2018-08-24 22:21:24,2018-10-16
50586,72f70be643bd8ca8da6be5b9e8435aa1,2018-08-22 14:54:48,2018-10-15
108425,f66b8bf378bdd2acca8604ef27039f1f,2018-08-09 08:24:53,2018-10-15
85887,c31a859e34e3adac22f376954e19b39d,2018-08-07 23:12:29,2018-10-10
91383,cf5c8d9f52807cb2d2f0a0ff54c478da,2018-08-24 13:04:05,2018-10-05
21680,31890e605bf108c3cbef0e15bd533614,2018-08-16 15:44:33,2018-10-05
105642,f0031384d36ff045ee1e866942d0fa7a,2018-08-23 00:07:26,2018-10-05


In [134]:
item_analysis["estimated_delivery_days"].describe(
    percentiles=[0.90, 0.95, 0.99, 0.995]
)

count    112650.000000
mean         23.835035
std           8.887306
min           2.008009
90%          34.388773
95%          38.595634
99%          51.061124
99.5%        55.513672
max         155.135463
Name: estimated_delivery_days, dtype: float64

In [135]:
(item_analysis["estimated_delivery_days"] > 60).sum()

np.int64(316)

In [136]:
(item_analysis["estimated_delivery_days"] > 60).mean() * 100

np.float64(0.2805148690634709)

In [137]:
missing_audit = (
    item_analysis.isna().sum()
    .sort_values(ascending=False)
)

missing_audit[missing_audit > 0]

order_delivered_customer_date    2454
delivery_delay_days              2454
is_delivered_late                2454
delivery_days                    2454
product_category_name_english    1627
product_description_lenght       1603
product_category_name            1603
product_name_lenght              1603
product_photos_qty               1603
order_delivered_carrier_date     1194
review_score_min                  942
review_score_avg                  942
review_score_max                  942
review_count                      942
product_length_cm                  18
product_weight_g                   18
product_height_cm                  18
product_width_cm                   18
order_approved_at                  15
payment_types                       3
payment_count                       3
total_payment_value                 3
max_payment_installments            3
dtype: int64

In [138]:
order_check = item_analysis.groupby("order_id").agg(
    actual_item_count=("order_item_id", "count"),
    actual_order_value=("item_total", "sum"),
    actual_order_freight=("freight_value", "sum")
)

order_check["item_count_match"] = (
    order_check["actual_item_count"] ==
    item_analysis.groupby("order_id")["order_item_count"].first()
)

order_check["value_match"] = (
    (order_check["actual_order_value"] -
     item_analysis.groupby("order_id")["order_total_value"].first()).abs() < 0.01
)

order_check["freight_match"] = (
    (order_check["actual_order_freight"] -
     item_analysis.groupby("order_id")["order_total_freight"].first()).abs() < 0.01
)

order_check[[
    "item_count_match",
    "value_match",
    "freight_match"
]].all()

item_count_match    True
value_match         True
freight_match       True
dtype: bool

In [141]:
seller_check = item_analysis.groupby("seller_id").agg(
    actual_item_count=("order_item_id", "count"),
    actual_order_count=("order_id", "nunique"),
    actual_total_sales=("item_total", "sum")
)

seller_check["item_count_match"] = (
    seller_check["actual_item_count"] ==
    item_analysis.groupby("seller_id")["seller_item_count"].first()
)

seller_check["order_count_match"] = (
    seller_check["actual_order_count"] ==
    item_analysis.groupby("seller_id")["seller_order_count"].first()
)

seller_check["sales_match"] = (
    (seller_check["actual_total_sales"] -
     item_analysis.groupby("seller_id")["seller_total_sales"].first()).abs() < 0.01
)

seller_check[[
    "item_count_match",
    "order_count_match",
    "sales_match"
]].all()

item_count_match     True
order_count_match    True
sales_match          True
dtype: bool

In [142]:
(
    (item_analysis["delivery_days"].isna()) ==
    (item_analysis["order_delivered_customer_date"].isna())
).all()

np.True_

In [147]:
(
    item_analysis.loc[
        item_analysis["delivery_delay_days"].notna(),
        "delivery_delay_days"
    ]
    ==
    (
        item_analysis.loc[
            item_analysis["delivery_delay_days"].notna(),
            "delivery_days"
        ]
        -
        item_analysis.loc[
            item_analysis["delivery_delay_days"].notna(),
            "estimated_delivery_days"
        ]
    )
).all()

np.True_

In [149]:
order_level = item_analysis[
    ["order_id", "customer_unique_id", "order_total_value"]
].drop_duplicates("order_id")

In [150]:
customer_spend_check = (
    order_level
    .groupby("customer_unique_id")
    .agg(actual_total_spend=("order_total_value", "sum"))
)

customer_spend_check["spend_match"] = (
    (customer_spend_check["actual_total_spend"] -
     item_analysis.groupby("customer_unique_id")["customer_total_spend"].first()
    ).abs() < 0.01
)

customer_spend_check["spend_match"].all()

np.True_